# Atelier Seaborn — Analyse exploratoire des capteurs IoT

## Contexte

Une entreprise possède plusieurs **bâtiments** équipés de **capteurs IoT**. Chaque capteur collecte régulièrement des informations sur :

- la **température**
- l'**humidité**
- la **pression**
- la **consommation énergétique**
- l'**état** du capteur (`OK`, `ALERTE`, `ERREUR`)
- le **bâtiment** et le **capteur** concernés
- la **date et l'heure** de la mesure

Après avoir utilisé **NumPy** pour manipuler les données numériques, **Pandas** pour importer, nettoyer et analyser le dataset, et **Matplotlib** pour réaliser des visualisations, cet atelier utilise **Seaborn** pour réaliser une **analyse exploratoire plus riche** des données.

## Sommaire

| Partie | Objectif |
|---|---|
| Setup | Installation, imports, chargement et vérification du dataset |
| Partie 1 | Distribution d'une variable avec `histplot()` |
| Partie 2 | Distribution d'une variable avec `kdeplot()` |
| Partie 3 | Distribution d'une variable avec `boxplot()` |
| Partie 4 | Distribution d'une variable avec `violinplot()` |
| Partie 5 | Comptage des catégories avec `countplot()` |
| Partie 6 | Relation entre deux variables avec `scatterplot()` |
| Partie 7 | Régression avec `regplot()` |
| Partie 8 | Régression avec `lmplot()` |
| Partie 9 | Corrélations et `heatmap()` |
| Partie 10 | Analyse multivariée avec `pairplot()` |
| Partie 11 | Sauvegarde des graphiques |
| Partie 12 | Bonus |

---


## Setup — Préparation de l'environnement

Avant toute analyse, on installe et importe les bibliothèques nécessaires, puis on charge le dataset dans un DataFrame `df` que l'on vérifie.

### 1. Installer et importer les bibliothèques

`seaborn`, `matplotlib`, `pandas` et `numpy` doivent être installés dans l'environnement (voir `requirements.txt` à la racine du projet, ou `pip install seaborn matplotlib pandas numpy`).

On importe ensuite :
- **pandas** (`pd`) pour manipuler le dataset sous forme de DataFrame
- **numpy** (`np`) pour les calculs numériques
- **matplotlib.pyplot** (`plt`) comme moteur de rendu graphique sous-jacent
- **seaborn** (`sns`) pour les visualisations statistiques de haut niveau

On configure aussi un style Seaborn agréable par défaut pour tous les graphiques de l'atelier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style et palette par défaut pour tous les graphiques de l'atelier
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (8, 5)

print("pandas   :", pd.__version__)
print("numpy    :", np.__version__)
print("matplotlib:", plt.matplotlib.__version__)
print("seaborn  :", sns.__version__)

### 2. Importer `mesures_capteurs.csv` dans le DataFrame `df`

Le fichier est stocké dans `../data/mesures_capteurs.csv` (chemin relatif au notebook, situé dans `notebooks/`).

In [ ]:
DATA_PATH = "../data/mesures_capteurs.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["date_heure"])
df.head()

### 3. Vérifier le contenu du DataFrame `df`

On inspecte la structure du dataset (dimensions, types de colonnes, valeurs manquantes) et quelques statistiques descriptives avant de commencer l'analyse exploratoire.

In [ ]:
print("Dimensions du dataset :", df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
# Valeurs manquantes par colonne
df.isna().sum()

In [ ]:
# Aperçu des catégories disponibles
print("Bâtiments :", sorted(df["batiment"].unique()))
print("Capteurs  :", sorted(df["id_capteur"].unique()))
print("États     :", df["etat"].unique())

> **Remarque :** on observe que la colonne `etat` contient quelques valeurs manquantes (`NaN`). Le nettoyage de ces valeurs sort du cadre de la Partie 1 et sera traité plus tard dans l'atelier si nécessaire.

---


## Partie 1 — Distribution d'une variable avec `histplot()`

L'**histogramme** découpe les valeurs d'une variable continue en classes (*bins*) et compte le nombre d'observations dans chaque classe. C'est l'outil de base pour visualiser la **distribution** d'une variable numérique : où sont concentrées les valeurs, la forme de la distribution, la présence de valeurs extrêmes, etc.

Ici, on étudie la distribution de la variable **température**.

### 1. Afficher la distribution des températures

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="temperature", kde=True)
plt.title("Distribution des températures")
plt.xlabel("Température (°C)")
plt.ylabel("Nombre de mesures")
plt.show()

### 2. Autour de quelle valeur les températures sont-elles concentrées ?

On peut lire la valeur centrale directement sur le graphique (pic de l'histogramme / de la courbe KDE), et la confirmer avec la moyenne et la médiane calculées par Pandas.

In [ ]:
temp_mean = df["temperature"].mean()
temp_median = df["temperature"].median()

print(f"Moyenne  : {temp_mean:.2f} °C")
print(f"Médiane  : {temp_median:.2f} °C")

**Réponse :** les températures sont concentrées autour de **{{mean}} °C** (moyenne ≈ médiane), ce qui correspond visuellement au pic de l'histogramme et de la courbe de densité (KDE).

*(Remplacer `{{mean}}` par la valeur affichée ci-dessus lors de la rédaction de la conclusion.)*

### 3. La distribution semble-t-elle symétrique ?

On compare visuellement la forme de l'histogramme de part et d'autre du pic, et on calcule le **coefficient d'asymétrie (skewness)** avec Pandas : une valeur proche de 0 indique une distribution symétrique.

In [ ]:
skewness = df["temperature"].skew()
print(f"Skewness (asymétrie) : {skewness:.3f}")

if abs(skewness) < 0.5:
    interpretation = "la distribution est globalement symétrique."
elif skewness > 0:
    interpretation = "la distribution est étalée vers la droite (asymétrie positive)."
else:
    interpretation = "la distribution est étalée vers la gauche (asymétrie négative)."

print("Interprétation :", interpretation)

**Réponse :** au vu de la forme de l'histogramme et d'une skewness proche de 0, la distribution des températures apparaît **globalement symétrique**, avec une allure proche d'une distribution normale (en cloche).

### 4. Existe-t-il des valeurs extrêmes ?

On observe les valeurs minimales et maximales, ainsi que les éventuelles barres isolées loin du corps principal de l'histogramme.

In [ ]:
print("Minimum :", df["temperature"].min())
print("Maximum :", df["temperature"].max())

# Détection de valeurs extrêmes avec la règle de l'écart interquartile (IQR)
q1 = df["temperature"].quantile(0.25)
q3 = df["temperature"].quantile(0.75)
iqr = q3 - q1
borne_basse = q1 - 1.5 * iqr
borne_haute = q3 + 1.5 * iqr

extremes = df[(df["temperature"] < borne_basse) | (df["temperature"] > borne_haute)]
print(f"Bornes IQR : [{borne_basse:.2f}, {borne_haute:.2f}]")
print(f"Nombre de valeurs extrêmes détectées : {len(extremes)}")
extremes[["id_mesure", "batiment", "temperature"]]

**Réponse :** à compléter après lecture des résultats ci-dessus — indiquer s'il existe des valeurs extrêmes selon la règle de l'IQR, et quels bâtiments/mesures sont concernés le cas échéant.

### 5. Modifier le nombre de classes (`bins`)

Le paramètre `bins` de `histplot()` contrôle le nombre de classes utilisées pour découper les valeurs. Un nombre de classes trop faible masque des détails de la distribution, un nombre trop élevé introduit du bruit. On compare plusieurs valeurs de `bins` côte à côte.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

for ax, n_bins in zip(axes, [10, 30, 60]):
    sns.histplot(data=df, x="temperature", bins=n_bins, kde=True, ax=ax)
    ax.set_title(f"bins = {n_bins}")
    ax.set_xlabel("Température (°C)")

axes[0].set_ylabel("Nombre de mesures")
plt.tight_layout()
plt.show()

**Observation :** un faible nombre de classes (`bins=10`) lisse fortement la distribution, tandis qu'un grand nombre de classes (`bins=60`) fait apparaître davantage de variations locales (bruit d'échantillonnage) sans changer la tendance générale.

### 6. Afficher la distribution des températures par bâtiment

On superpose ou distingue les distributions de température **par bâtiment** grâce au paramètre `hue`, afin de comparer les profils thermiques des différents bâtiments.

In [ ]:
plt.figure(figsize=(9, 5.5))
sns.histplot(data=df, x="temperature", hue="batiment", kde=True, element="step", stat="density", common_norm=False)
plt.title("Distribution des températures par bâtiment")
plt.xlabel("Température (°C)")
plt.ylabel("Densité")
plt.show()

On peut aussi afficher un histogramme séparé par bâtiment grâce à `col` (facettes) pour une lecture plus détaillée :

In [ ]:
g = sns.displot(
    data=df, x="temperature", col="batiment", col_wrap=2,
    kde=True, height=3.5, aspect=1.2
)
g.set_axis_labels("Température (°C)", "Nombre de mesures")
g.set_titles("Bâtiment {col_name}")
plt.show()

**Conclusion Partie 1 :** l'histogramme (`histplot()`) permet de visualiser la forme générale de la distribution de la température : sa valeur centrale, sa symétrie, la présence de valeurs extrêmes, et les différences de profil entre bâtiments grâce à `hue` ou aux facettes (`col`).

---


## Partie 2 — Distribution d'une variable avec `kdeplot()`

Le **KDE** (*Kernel Density Estimate*) estime une courbe de densité continue à partir des observations, sans le découpage en classes de l'histogramme. Il permet de lire la forme de la distribution de façon plus lisse, ce qui facilite la comparaison entre plusieurs groupes.

### 1. Afficher la distribution des températures

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x="temperature", fill=True)
plt.title("Distribution des températures (KDE)")
plt.xlabel("Température (°C)")
plt.ylabel("Densité")
plt.show()

### 2. Afficher la distribution des températures par bâtiment

On distingue les courbes de densité **par bâtiment** avec `hue`, pour comparer visuellement leurs profils de température.

In [ ]:
plt.figure(figsize=(9, 5.5))
sns.kdeplot(data=df, x="temperature", hue="batiment", fill=True, common_norm=False, alpha=0.3)
plt.title("Distribution des températures par bâtiment (KDE)")
plt.xlabel("Température (°C)")
plt.ylabel("Densité")
plt.show()

**Conclusion Partie 2 :** le `kdeplot()` offre une vue lissée et continue de la distribution, plus facile à superposer entre plusieurs bâtiments que des histogrammes. Il confirme les observations faites avec `histplot()` (valeur centrale, symétrie) tout en rendant les comparaisons entre groupes plus lisibles.

---


## Partie 3 — Distribution d'une variable avec `boxplot()`

La **boîte à moustaches** (*box plot*) résume une distribution avec ses quartiles : la boîte représente l'écart interquartile (Q1 à Q3), la ligne centrale la médiane, les moustaches l'étendue des valeurs "normales", et les points isolés les valeurs extrêmes (*outliers*). C'est l'outil idéal pour comparer rapidement plusieurs groupes.

### 1. Afficher la distribution des températures

In [ ]:
plt.figure(figsize=(6, 5.5))
sns.boxplot(data=df, y="temperature")
plt.title("Distribution des températures (box plot)")
plt.ylabel("Température (°C)")
plt.show()

### 2. Afficher la distribution des températures par bâtiment

In [ ]:
plt.figure(figsize=(9, 5.5))
sns.boxplot(data=df, x="batiment", y="temperature", order=sorted(df["batiment"].unique()))
plt.title("Distribution des températures par bâtiment (box plot)")
plt.xlabel("Bâtiment")
plt.ylabel("Température (°C)")
plt.show()

### 3. Pour chaque bâtiment : médiane, dispersion et valeurs extrêmes

On calcule avec Pandas, pour chaque bâtiment :
- **a) la médiane** de la température
- **b) la dispersion**, mesurée par l'écart interquartile (IQR = Q3 − Q1)
- **c) les valeurs extrêmes**, détectées avec la règle IQR (au-delà de `Q1 - 1.5×IQR` ou `Q3 + 1.5×IQR`)

In [ ]:
def stats_boite(groupe):
    q1 = groupe.quantile(0.25)
    q3 = groupe.quantile(0.75)
    iqr = q3 - q1
    borne_basse = q1 - 1.5 * iqr
    borne_haute = q3 + 1.5 * iqr
    n_extremes = ((groupe < borne_basse) | (groupe > borne_haute)).sum()
    return pd.Series({
        "mediane": groupe.median(),
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "borne_basse": borne_basse,
        "borne_haute": borne_haute,
        "nb_valeurs_extremes": n_extremes,
    })

stats_par_batiment = df.groupby("batiment")["temperature"].apply(stats_boite).unstack()
stats_par_batiment

**Réponse :**
- **a) Médiane** : voir la colonne `mediane` du tableau ci-dessus — c'est la ligne centrale de chaque boîte sur le graphique.
- **b) Dispersion** : mesurée par la colonne `iqr` (hauteur de la boîte) — plus l'IQR est grand, plus le bâtiment a des températures dispersées.
- **c) Valeurs extrêmes** : le nombre de valeurs extrêmes par bâtiment est donné par `nb_valeurs_extremes`, correspondant aux points isolés au-delà des moustaches sur le box plot.

*(Compléter avec les valeurs numériques observées dans le tableau pour rédiger la conclusion.)*

### 4. Quels bâtiments ont les valeurs les plus extrêmes ?

On identifie le(s) bâtiment(s) ayant la plus grande dispersion (IQR) et le plus grand nombre de valeurs extrêmes.

In [ ]:
batiment_plus_disperse = stats_par_batiment["iqr"].idxmax()
batiment_plus_extremes = stats_par_batiment["nb_valeurs_extremes"].idxmax()

print(f"Bâtiment avec la plus grande dispersion (IQR)      : {batiment_plus_disperse}")
print(f"Bâtiment avec le plus de valeurs extrêmes détectées : {batiment_plus_extremes}")

stats_par_batiment.sort_values("nb_valeurs_extremes", ascending=False)

**Conclusion Partie 3 :** le `boxplot()` résume efficacement chaque distribution par sa médiane, sa dispersion (IQR) et ses valeurs extrêmes, et permet de comparer rapidement plusieurs bâtiments côte à côte pour repérer celui qui se distingue le plus.

---


## Partie 4 — Distribution d'une variable avec `violinplot()`

Le **violin plot** combine un box plot et un KDE : il affiche à la fois les quartiles/médiane (comme le box plot) et la forme complète de la densité de part et d'autre de l'axe (comme le KDE). Il permet de voir si une distribution est unimodale, bimodale, etc.

### 1. Afficher la distribution des températures

In [ ]:
plt.figure(figsize=(6, 5.5))
sns.violinplot(data=df, y="temperature")
plt.title("Distribution des températures (violin plot)")
plt.ylabel("Température (°C)")
plt.show()

### 2. Afficher la distribution des températures par bâtiment

In [ ]:
plt.figure(figsize=(9, 5.5))
sns.violinplot(data=df, x="batiment", y="temperature", order=sorted(df["batiment"].unique()))
plt.title("Distribution des températures par bâtiment (violin plot)")
plt.xlabel("Bâtiment")
plt.ylabel("Température (°C)")
plt.show()

### 3. Quelles informations supplémentaires le violin plot permet-il de visualiser par rapport au box plot ?

**Réponse :** contrairement au box plot qui ne montre que les quartiles, la médiane et les valeurs extrêmes, le violin plot affiche en plus la **forme complète de la densité** de la distribution (largeur du violon à chaque niveau de température). Il permet ainsi de détecter des distributions **multimodales** (plusieurs pics), des zones de concentration des valeurs à l'intérieur même de l'écart interquartile, ou des asymétries locales que le box plot, plus résumé, ne peut pas révéler.

**Conclusion Partie 4 :** le `violinplot()` enrichit le `boxplot()` en combinant synthèse statistique (quartiles, médiane) et forme détaillée de la distribution (densité), au prix d'une lecture un peu moins immédiate pour les novices.

---


## Partie 5 — Comptage des catégories avec `countplot()`

Le `countplot()` affiche le nombre d'observations pour chaque catégorie d'une variable qualitative. Il est utilisé ici pour analyser les variables **état** (`etat`) et **bâtiment** (`batiment`).

### 1. Afficher le nombre de mesures par état

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="etat", order=df["etat"].value_counts().index)
plt.title("Nombre de mesures par état")
plt.xlabel("État")
plt.ylabel("Nombre de mesures")
plt.show()

### 2. Afficher le nombre de mesures par bâtiment

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="batiment", order=sorted(df["batiment"].unique()))
plt.title("Nombre de mesures par bâtiment")
plt.xlabel("Bâtiment")
plt.ylabel("Nombre de mesures")
plt.show()

### 3. États des capteurs par bâtiment

On croise `batiment` et `etat` avec `hue` pour identifier le bâtiment ayant le plus de mesures en **OK**, en **ALERTE**, et en **ERREUR**.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(
    data=df, x="batiment", hue="etat",
    order=sorted(df["batiment"].unique()),
    hue_order=["OK", "ALERTE", "ERREUR"],
)
plt.title("États des capteurs par bâtiment")
plt.xlabel("Bâtiment")
plt.ylabel("Nombre de mesures")
plt.legend(title="État")
plt.show()

In [ ]:
tableau_croise = pd.crosstab(df["batiment"], df["etat"])
tableau_croise

In [ ]:
batiment_plus_ok = tableau_croise["OK"].idxmax()
batiment_plus_alerte = tableau_croise["ALERTE"].idxmax()
batiment_plus_erreur = tableau_croise["ERREUR"].idxmax()

print(f"a) Bâtiment avec le plus de mesures en OK     : {batiment_plus_ok}")
print(f"b) Bâtiment avec le plus de mesures en ALERTE : {batiment_plus_alerte}")
print(f"c) Bâtiment avec le plus de mesures en ERREUR : {batiment_plus_erreur}")

**Conclusion Partie 5 :** le `countplot()` permet de visualiser rapidement la répartition des catégories (états, bâtiments), et le croisement avec `hue` (confirmé par `pd.crosstab()`) permet d'identifier précisément quel bâtiment concentre le plus de mesures dans chaque état.

---


## Partie 6 — Relation entre deux variables avec `scatterplot()`

Le **nuage de points** (*scatter plot*) permet d'étudier visuellement la relation entre deux variables numériques. On étudie ici la relation entre la **température** et la **consommation énergétique**.

### 1. Étudier graphiquement la relation entre température et consommation

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="temperature", y="consommation")
plt.title("Relation entre température et consommation")
plt.xlabel("Température (°C)")
plt.ylabel("Consommation")
plt.show()

### 2. Étudier graphiquement la relation entre température et consommation par bâtiment

On distingue les bâtiments avec `hue` pour voir si la relation température/consommation diffère d'un bâtiment à l'autre.

In [ ]:
plt.figure(figsize=(9, 6.5))
sns.scatterplot(data=df, x="temperature", y="consommation", hue="batiment")
plt.title("Relation entre température et consommation par bâtiment")
plt.xlabel("Température (°C)")
plt.ylabel("Consommation")
plt.legend(title="Bâtiment")
plt.show()

### 3. Modifier la taille des points

Le paramètre `s` fixe une taille unique pour tous les points. On peut aussi utiliser `size` pour faire varier la taille des points en fonction d'une troisième variable numérique, ici l'**humidité**.

In [ ]:
# a) taille de points fixe et plus grande
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="temperature", y="consommation", hue="batiment", s=100)
plt.title("Relation température / consommation — points agrandis (s=100)")
plt.xlabel("Température (°C)")
plt.ylabel("Consommation")
plt.legend(title="Bâtiment")
plt.show()

In [ ]:
# b) taille des points variable selon l'humidité
plt.figure(figsize=(9, 6.5))
sns.scatterplot(data=df, x="temperature", y="consommation", hue="batiment", size="humidite", sizes=(20, 200))
plt.title("Relation température / consommation — taille proportionnelle à l'humidité")
plt.xlabel("Température (°C)")
plt.ylabel("Consommation")
plt.legend(title="Bâtiment / Humidité", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

**Conclusion Partie 6 :** le `scatterplot()` permet d'observer visuellement la relation entre température et consommation, de la décliner par bâtiment avec `hue`, et d'encoder une troisième variable (l'humidité) via la taille des points avec `size`.

---


## Partie 7 — Régression avec `regplot()`

Le `regplot()` superpose au nuage de points une **droite de régression linéaire** (avec son intervalle de confiance), ce qui permet d'évaluer visuellement s'il existe une tendance globale entre deux variables numériques.

### Étudier graphiquement une tendance entre température et consommation

In [ ]:
plt.figure(figsize=(8, 6))
sns.regplot(data=df, x="temperature", y="consommation", scatter_kws={"alpha": 0.5})
plt.title("Tendance entre température et consommation")
plt.xlabel("Température (°C)")
plt.ylabel("Consommation")
plt.show()

**Conclusion Partie 7 :** la pente de la droite de régression indique le sens de la tendance (croissante ou décroissante) entre température et consommation, tandis que la dispersion des points autour de la droite et la largeur de l'intervalle de confiance renseignent sur la force de cette relation linéaire.

---
